# Embeddings from scratch

SETUP (one time):
pip install sentence-transformers numpy scikit-learn
 

In [1]:
import numpy as np
from sentence_transformers import SentenceTransformer

c:\AAA\Personal\AIDE_Prep\CRag\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Step 1: Embedding of sentence

In [2]:
model = SentenceTransformer('models/all-MiniLM-L6-v2')

sentence = "Machine learning is a subset of artificial intelligence"
embedding = model.encode(sentence)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4955.82it/s]
BertModel LOAD REPORT from: models/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
print(f'\nSentence: {sentence}')
print(f'Embedding shape: {embedding.shape}')
print(f'First 10 values: {embedding[:10].round(4)}')


Sentence: Machine learning is a subset of artificial intelligence
Embedding shape: (384,)
First 10 values: [-0.0461 -0.0043  0.0698  0.0355  0.0485 -0.0302  0.0016 -0.0095 -0.0514
 -0.0039]


In [4]:
print(f'Min: {embedding.min(): .4f}, Max:{embedding.max():.4f}')

Min: -0.1771, Max:0.1363


In [5]:
# KEY INSIGHT: Har sentence ek 384-dimensional point ban gaya.
# Ab hum sentences ko "compare" kar sakte hain mathematically!
 

# Step 2: Cosine Similarity - 'Similarity of 2 sentences'

In [6]:
def cosine_similarity(a, b):
    """
    Cosine similarity = dot(a, b) / (||a|| * ||b||)
    
    Range: -1 to +1
    +1 = exactly same direction (same meaning)
     0 = unrelated
    -1 = opposite meaning
    
   """

    dot_product = np.dot(a, b)
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)
    return dot_product / (norm_a * norm_b)

In [7]:
# Compare 3 sentences
s1 = "Machine learning is a branch of AI"
s2 = "Deep learning uses neural networks for AI tasks"
s3 = "I love eating pizza on weekends"

e1 = model.encode(s1)
e2 = model.encode(s2)
e3 = model.encode(s3)

sim_12 = cosine_similarity(e1, e2)
sim_13 = cosine_similarity(e1, e3)
sim_23 = cosine_similarity(e2, e3)

In [8]:
print(f"\n'{s1}'")
print(f"  vs '{s2}'")
print(f"  Similarity: {sim_12:.4f}  ← SIMILAR TOPIC (AI/ML)")
 
print(f"\n'{s1}'")
print(f"  vs '{s3}'")
print(f"  Similarity: {sim_13:.4f}  ← UNRELATED")
 
print(f"\n'{s2}'")
print(f"  vs '{s3}'")
print(f"  Similarity: {sim_23:.4f}  ← UNRELATED")


'Machine learning is a branch of AI'
  vs 'Deep learning uses neural networks for AI tasks'
  Similarity: 0.5380  ← SIMILAR TOPIC (AI/ML)

'Machine learning is a branch of AI'
  vs 'I love eating pizza on weekends'
  Similarity: -0.0039  ← UNRELATED

'Deep learning uses neural networks for AI tasks'
  vs 'I love eating pizza on weekends'
  Similarity: 0.0553  ← UNRELATED


# Step 3: Semantic Search from scratch

In [9]:
knowledge_base = [
    "Python is a popular programming language for data science",
    "TensorFlow and PyTorch are deep learning frameworks",
    "Kubernetes orchestrates containerized applications at scale",
    "RAG combines retrieval with generation for accurate AI responses",
    "Docker packages applications into portable containers",
    "Vector databases store embeddings for similarity search",
    "Fine-tuning adapts pre-trained models to specific domains",
    "Kafka is used for real-time event streaming pipelines",
    "FastAPI is a modern Python web framework for building APIs",
    "Transformers use attention mechanism for sequence modeling",
]



In [10]:
# Encode all documents
print('\nEncoding knowledge base')
kb_embeddings = model.encode(knowledge_base)
print(f'Knowledge base shape: {kb_embeddings.shape}')


Encoding knowledge base
Knowledge base shape: (10, 384)


In [12]:
# User asks a question
query = 'How do i deploy machine learning models?'
query_embedding = model.encode(query)

In [13]:
# Find most similar documents
print(f'\nQuery: {query}')
print(f'\nRanked results (by cosine similarity):')

similarities = []
for i, doc_emb in enumerate(kb_embeddings):
    sim = cosine_similarity(query_embedding, doc_emb)
    similarities.append((sim, i))

# Sort by similarity
similarities.sort(reverse=True)

for rank, (sim, idx) in enumerate(similarities, 1):
    marker = "  ← TOP RESULT!" if rank == 1 else ""
    print(f"  #{rank} [{sim:.4f}] {knowledge_base[idx]}{marker}")


Query: How do i deploy machine learning models?

Ranked results (by cosine similarity):
  #1 [0.3571] Fine-tuning adapts pre-trained models to specific domains  ← TOP RESULT!
  #2 [0.3119] TensorFlow and PyTorch are deep learning frameworks
  #3 [0.2755] Kubernetes orchestrates containerized applications at scale
  #4 [0.2117] FastAPI is a modern Python web framework for building APIs
  #5 [0.1464] Docker packages applications into portable containers
  #6 [0.1193] Python is a popular programming language for data science
  #7 [0.1174] Vector databases store embeddings for similarity search
  #8 [0.1034] RAG combines retrieval with generation for accurate AI responses
  #9 [0.0826] Kafka is used for real-time event streaming pipelines
  #10 [0.0750] Transformers use attention mechanism for sequence modeling


# Step 4: Top K retrieval

In [ ]:
def retrieve(query: str, k: int=3) -> list[str]:
    q_emb = model.encode(query)
    scores = []
    for i, doc_emb in enumerate(kb_embeddings):
        sim = cosine_similarity(q_emb, doc_emb)
        scores.append((sim, i))

    scores.sort(reverse=True)

    results = []
    for sim, idx in scores[:k]:
        results.append({
            "text": knowledge_base[idx],
            "score": round(float(sim), 4)
        })
    return results




In [16]:
# Test with different queries
test_queries = [
    "What frameworks are used for deep learning?",
    "How to build REST APIs in Python?",
    "What is retrieval augmented generation?",
    "How does real-time data processing work?",
]

In [17]:
for query in test_queries:
    print(f"\nQuery: '{query}'")
    results = retrieve(query, k=3)
    for i, r in enumerate(results, 1):
        print(f"  {i}. [{r['score']}] {r['text']}")


Query: 'What frameworks are used for deep learning?'
  1. [0.7235] TensorFlow and PyTorch are deep learning frameworks
  2. [0.331] Fine-tuning adapts pre-trained models to specific domains
  3. [0.2973] FastAPI is a modern Python web framework for building APIs

Query: 'How to build REST APIs in Python?'
  1. [0.6574] FastAPI is a modern Python web framework for building APIs
  2. [0.4031] Python is a popular programming language for data science
  3. [0.1679] Docker packages applications into portable containers

Query: 'What is retrieval augmented generation?'
  1. [0.4865] RAG combines retrieval with generation for accurate AI responses
  2. [0.2652] Transformers use attention mechanism for sequence modeling
  3. [0.2643] Fine-tuning adapts pre-trained models to specific domains

Query: 'How does real-time data processing work?'
  1. [0.5045] Kafka is used for real-time event streaming pipelines
  2. [0.2531] Python is a popular programming language for data science
  3. [0.2211] 

# Step 5: Why naive similarity search isn't enough

In [ ]:
# Problem 1
q_pos = "Python is good for machine learning"
q_neg = "Python is terrible for machine learning"
e_pos = model.encode(q_pos)
e_neg = model.encode(q_neg)


In [19]:
sim_neg = cosine_similarity(e_pos, e_neg)
print(f'\n[Problem: Negation]')
print(f" '{q_pos}")
print(f"  vs '{q_neg}'")
print(f"  Similarity: {sim_neg:.4f}  ← Way too high! Opposite meaning par similar vector!")
print(f"  WHY: Embeddings capture TOPIC, not SENTIMENT perfectly.")
 


[Problem: Negation]
 'Python is good for machine learning
  vs 'Python is terrible for machine learning'
  Similarity: 0.8608  ← Way too high! Opposite meaning par similar vector!
  WHY: Embeddings capture TOPIC, not SENTIMENT perfectly.


In [20]:
# Problem 2: Short vs long text
q_short = "ML"
q_long = "Machine learning is a comprehensive field of study within artificial intelligence"
e_short = model.encode(q_short)
e_long = model.encode(q_long)
 
sim_len = cosine_similarity(e_short, e_long)
print(f"\n[PROBLEM: Length mismatch]")
print(f"  '{q_short}' vs long description")
print(f"  Similarity: {sim_len:.4f}  ← Might be lower than expected")
print(f"  WHY: Very short text has less semantic signal.")
 


[PROBLEM: Length mismatch]
  'ML' vs long description
  Similarity: 0.2042  ← Might be lower than expected
  WHY: Very short text has less semantic signal.


In [21]:
# Problem 3: Out-of-domain
q_ood = "quantum entanglement in physics"
results_ood = retrieve(q_ood, k=2)
print(f"\n[PROBLEM: Out-of-domain query]")
print(f"  Query: '{q_ood}'")
print(f"  Best match: [{results_ood[0]['score']}] {results_ood[0]['text']}")
print(f"  WHY: Model will ALWAYS return something, even if nothing is relevant!")
print(f"  SOLUTION: Set a minimum similarity threshold (e.g., 0.3)")
 


[PROBLEM: Out-of-domain query]
  Query: 'quantum entanglement in physics'
  Best match: [0.1255] TensorFlow and PyTorch are deep learning frameworks
  WHY: Model will ALWAYS return something, even if nothing is relevant!
  SOLUTION: Set a minimum similarity threshold (e.g., 0.3)
